# Toehold switch — single input

Manual test harness for `engine.gates.toehold.ToeholdGate`. Run the cells top to
bottom. Scientific methods raise `NotImplementedError` until the Step 5 bodies
land — `fx.attempt(...)` prints `pending Step 5` for those instead of a traceback.

## The mechanism

A toehold switch is an mRNA that will not translate itself until told to. It folds
into a hairpin: the ribosome binding site sits in the accessible **loop**, the
**start codon** is buried in the **stem**, and a single-stranded **toehold** hangs
off the 5' end. The trigger pairs with the toehold, unzips the stem, frees the start
codon, and translation begins.

A good design keeps the OFF hairpin dark (low leakage) while making trigger binding
more favourable still (high dynamic range). Those pull against each other — which is
why design is a search, not a formula.

## Setup

In [1]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

engine package root: /home/zivbental/coding/igem/cernal_software/src


PosixPath('/home/zivbental/coding/igem/cernal_software/src')

## Build the gate

In [2]:
host = fx.Host.ECOLI          # ECOLI | YEAST | HUMAN are all supported
gate = fx.toehold(host=host)
fx.describe_gate(gate)

ToeholdGate   <ToeholdGate toehold-0.1.0-stub>
  name             toehold
  version          0.1.0-stub
  kind             toehold
  label            Toehold Riboswitch
  description      Translational control · pre-mRNA
  max_inputs       1
  available        True
  supported_hosts  ['ecoli', 'human', 'yeast']
  instance host    ecoli   supports(host)=True


## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [3]:
triggers = fx.sample_trigger_set(n_activators=1)
constraints = fx.sample_constraints(max_switch_length=200)

for t in triggers.activators:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt  GC {t.gc_content:.0f}%')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

trig-000001  thrA    AGACUUUCAAAGAUAUGCUGGGUAGAGGUCGAGGUU  36 nt  GC 44%
arity: 1 | logic: single-input


## `required_tools()` — checked before a run starts

Implemented. A missing dependency should fail here, fast, not part-way through
an expensive run.

In [4]:
fx.attempt('required_tools', gate.required_tools)

ok :: required_tools
  [ToolRequirement(name='ViennaRNA', version='2.7', optional=False)]


[ToolRequirement(name='ViennaRNA', version='2.7', optional=False)]

## `is_compatible()` — cheap gate/trigger check *(Step 5)*

Runs for every trigger set against every family. Cheap checks only — arity, host,
trigger separation, trigger length window. Nothing here should fold.

In [5]:
fx.attempt('is_compatible', lambda: gate.is_compatible(triggers, constraints))

pending Step 5 :: is_compatible  ->  Step 5


## `generate_designs()` — candidate switches *(Step 5)*

Yields, potentially tens of thousands of designs across a run, so materialise a
handful with `list(...)` here.

In [6]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

pending Step 5 :: generate_designs  ->  Step 5


## `evaluate_design()` — raw metrics for one design *(Step 5)*

Returns **raw** values keyed by metric name — no normalising, weighting or
filtering (that is `engine.scoring`'s job). `None` for a metric that could not be
computed, never a sentinel.

In [7]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

pending Step 5 :: evaluate_design  ->  Step 5


## `emit_sequence()` and `describe()` — output helpers

These two are implemented today. `generate_designs()` is not, so
`fx.sample_design(...)` hands us a plausible `GateDesign` to call them on.

In [8]:
design = fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

design_id   demo-000001
gate_kind   toehold
length      63 nt
emit_sequence: AUGGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCU
describe     : single-input toehold gate on thrA


## Switching in real folding

Everything above runs against `fx.StubFoldEngine` — deterministic, fake, no
ViennaRNA. For genuine structure predictions, pass `real_fold=True` when you build
the gate (needs `import RNA` to work in this environment):

```python
gate = fx.toehold(host=host, real_fold=True)
folder = fx.fold_engine(real=True)
folder.mfe('GGGAAACCCUUUGGGAAACCC')   # -> FoldResult(structure, energy)
```